# Reviewer Full Evaluation on RRUFF-325

This notebook runs the public `RRUFF-325` calibration evaluation path for a released checkpoint.

It is intentionally narrower than the full paper pipeline:

- one checkpoint
- one benchmark (`RRUFF-325`)
- one evaluation module (`evaluate_calibration_metrics`)

Required local inputs:

- a compatible checkpoint under `external/checkpoints/`
- the `RRUFF-325` HDF5 benchmark file
- the compact aggregate prior JSON used for Bayesian calibration


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

CHECKPOINT_PATH = Path(os.environ.get('CHECKPOINT_PATH', REPO_ROOT / 'external' / 'checkpoints' / 'xrd_model_82ept35h_best.pth'))
CONFIG_PATH = Path(os.environ.get('CONFIG_PATH', REPO_ROOT / 'configs' / 'final_mixed_2500k_dualsource.json'))
RRUFF_325_H5 = Path(os.environ.get('RRUFF_325_H5', REPO_ROOT / 'external' / 'benchmarks' / 'RRUFF_usable_plus_recoverable_325_with_labels_maxnorm.hdf5'))
PRIOR_DATA_PATH = Path(os.environ.get('PRIOR_DATA_PATH', REPO_ROOT / 'results' / 'reviewer' / 'ext_group_priors.json'))
OUTDIR = Path(os.environ.get('OUTDIR', REPO_ROOT / 'results' / 'reviewer_rerun'))
OUTDIR.mkdir(parents=True, exist_ok=True)

print('Repo root      :', REPO_ROOT)
print('Checkpoint     :', CHECKPOINT_PATH)
print('Config         :', CONFIG_PATH)
print('RRUFF-325 HDF5 :', RRUFF_325_H5)
print('Prior source   :', PRIOR_DATA_PATH)
print('Output dir     :', OUTDIR)


In [ ]:
missing = [p for p in [CHECKPOINT_PATH, CONFIG_PATH, RRUFF_325_H5, PRIOR_DATA_PATH] if not p.exists()]
if missing:
    missing_text = '\n'.join(str(p) for p in missing)
    raise FileNotFoundError(
        'Missing required input(s):\n'
        f'{missing_text}\n\n'
        'See docs/EVALUATION.md and reproducibility/dataset_manifest.csv for the expected artifact names.'
    )


In [ ]:
output_json = OUTDIR / 'rruff325_calibration.json'
cmd = [
    sys.executable,
    '-m', 'paper_ai_diffraction.eval.evaluate_calibration_metrics',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--config', str(CONFIG_PATH),
    '--eval-data-path', str(RRUFF_325_H5),
    '--prior-data-path', str(PRIOR_DATA_PATH),
    '--aux-temperature', '5.0',
    '--num-workers', '4',
    '--bootstrap', '200',
    '--output-json', str(output_json),
]

print('Running:')
print(' '.join(cmd))
env = os.environ.copy()
src_path = str(SRC_PATH)
env['PYTHONPATH'] = src_path + os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else src_path
completed = subprocess.run(cmd, cwd=REPO_ROOT, env=env, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f'Calibration evaluation failed with exit code {completed.returncode}')


In [ ]:
result = json.loads(output_json.read_text())
uncal = result['uncalibrated_aux']['point_estimate']
cal = result['calibrated_bayesian_aux']['point_estimate']

print('Uncalibrated auxiliary head:')
print(json.dumps(uncal, indent=2))
print('\nCalibrated Bayesian auxiliary head:')
print(json.dumps(cal, indent=2))

expected_top1 = 0.1046
expected_top5 = 0.4369
print('\nPaper reference for final mixed champion on RRUFF-325:')
print(f'expected Top-1 ~= {expected_top1:.4f}')
print(f'expected Top-5 ~= {expected_top5:.4f}')
print(f'observed Top-1 = {cal["top1"]:.4f}')
print(f'observed Top-5 = {cal["top5"]:.4f}')
